# [Crategraph](https://github.com/unimelbmdap/cdl2-rocrates) Cheat Sheet

**Crategraph basics**

> Inspired by Franz Diebold's Polars cheat sheet (cheat sheet will eventually look similar).


## Before we begin...

A `crate` from Crategraph Python package is a **`Graph`** consisting of two layers of data:

```
        ENTITY (node)              RELATIONSHIP (edge)
        ┌────────────┐  --type-->  ┌──────────────┐
        │ .id .types │ ══════════> │ .type         │
        │ .properties│             │ .source/.target│
        │ .name      │             │ .properties    │
        └────────────┘             └──────────────┘
              │                            │
     filter by PROPERTIES         filter by STRUCTURE
        crate.where(...)            crate.select(...)
```

**It's important to keep in mind:** the layer you query must match the method.

| Filtering by… | Method to use | Example |
|---|---|---|
| a value *on a node* (name, year, nationality) | `where` | `crate.where(name="Smith")` |
| a *type* or *edge* (entity_types, relationship_types) | `select` | `crate.select(relationship_types="Primary")` |

For example, running `where(relationship_types="Primary")` returns **0** (it looks for a node *property* literally called `relationship_types`, which doesn't exist).

**Remember:** every filter/transform returns a **new `Graph`**, so they chain.

We will explore crategraph package together below

#### Case Study: The University of Melbourne Perpetual Calendar (UMPC) RO-Crate

**Source:** https://umpc.esrc.unimelb.edu.au

## Import/Install

Make sure crategraph package is up to date and import it:

In [ ]:
!git pull

## Load a graph


In [ ]:
from crategraph import Crate

Load your graph (RO-Crate) from a local directory pointing at ro-crate-metadata.json
<br>
crate = Crate("path/to/your/crate/")
<br>
Crate represents a Graph (N entities, M relationships)

In [ ]:
crate = Crate(
    "./data/UMPC-ro-crate"
)

In [ ]:
# Options:
#   Crate(p1, p2)              -> load several crates (IDs prefixed by dir name)
#   Crate(path, include_root=True)
#   Crate(path, inline_relations=False)   # only reified Relationship entities

You can get a quick "feel" about your crate using these methods:

Summary of a graph including a complete list of entity types and relationship counts:

In [ ]:
crate.summary()

Stats on graph profile (density, components, connectivity, connections distribution etc):

In [ ]:
crate.profile()

Data-quality and imperfections of RO-Crate matadata:

## Scratchpad/Recipes

Your own experiments below.


In [ ]:
crate.coverage(
    inline_relations=True, min_occurrences=1
)  # Unsure if this is useful to show/what is the difference between these two operations
crate.coverage()

Visual representation (snapshot) of the graph:

In [ ]:
crate.glimpse()

## Entities (nodes)

Explore items in the crate: people, files, places. Each of them has `.properties` (a data
dictionaty) and attribute `.types`.


Get your bearings - what does this graph contain?

In [ ]:
crate.entities  # list[Entity] of all nodes

You can explore an individual entity from this list:

In [ ]:
crate.entities[777]

Get all file entities:

In [ ]:
crate.files

Get a sorted list of all entity type names present in the graph: 

In [ ]:
list(crate.types)

Get a number of entities (also visisble after calling .profile()):

In [ ]:
len(crate)

**Inspecting an indivisual entity:**

In [ ]:
e = crate.entities[777]

In [ ]:
print(
    e
)  # I will omit e.id, e.name, e.type, e.types since a user will most likely get all they need
# from the next attribute lookup

In [ ]:
e.properties

Get a specific entity by id:

In [ ]:
crate.get("#E001213")

Get an entity and its live reference to the graph. EntityView class is used in annotate_entities() method.

In [ ]:
e2 = crate.entity_view("#E001213")

You can call help() around an object, if things become unclear:

In [ ]:
help(e2)

In [ ]:
e2.properties

**Discover all property keys** 


If you want to explore property values in your graph, these are the ways to approach it:

In [ ]:
all_names = set()
for e in crate.entities:
    all_names.update(e.properties)
sorted(all_names)

In [ ]:
sorted({k for e in crate.entities for k in e.properties})

**Exporting results to pandas / polars**

If you want to visualise/export your data to a Dataframe, import pandas (or polars if you prefer) first:

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame(crate.entity_records())  # all enties in the graph (one row per entity)

Export this data into an Excel

In [ ]:
pd.DataFrame(crate.entity_records()).sort_values("startDate").to_csv("allrecords.csv", index=False)

In [ ]:
!open "allrecords.csv"

In [ ]:
pd.DataFrame(
    crate.entity_records(columns=["id", "name", "type"])
)  # filter out by property keys of interest

Return a count of distinct values for any property of an entity (sorted by highest count, descending):

In [ ]:
crate.entity_counts("function")

In [ ]:
crate.entity_counts("type")

**Filter a graph property**

Find all the lawyers in the graph:

In [ ]:
crate.where(function="Lawyer")

Find all the entries for this date period (inclusive);

In [ ]:
crate.where(startDate=(1870, 1900))

**Enrich**


Select all people born im Victoria in this crate. You can start with exploring types of entities in the graph: list(crate.types) to find where that is represented.

First, see what Victoria is linked to:  

In [ ]:
crate.annotate_entities(place=lambda e: e.related("birthPlace").first("name")).entity_counts(
    "place"
)

In [ ]:
crate.annotate_entities(state=lambda e: e.related("birthState").first("name")).entity_counts(
    "state"
)

Now that we know Victoria is a property of a birthState, we can add a new property to entities:

In [ ]:
crate2 = crate.annotate_entities(
    is_victorian=lambda e: e.related("birthState").first("name") == "Victoria"
)

In [ ]:
len(crate2.where(is_victorian=True))

Find out the top hub entities in this graph (10 by default, adjust n to your preference).

In [ ]:
crate.most_connected(n=3)

##  Relationships (edges)

Directed links between entities. Each has a `.type`, `.source`, `.target`.

**Look around**

Have a look at what relationships exist in the graph:

In [ ]:
# crate.relationships  # returns a list of all relationship objects in the graph
list(crate.relationship_types)  # returns distinct relationship type names

Visualise a single relationship object in a graph:

In [ ]:
print(crate.relationships[111])

**Count and export**

Return a count of how many relationships exist for each relationship type (sorted by highest count, descending):

In [ ]:
crate.relationship_counts("type")

In [ ]:
import pandas as pd

pd.DataFrame(crate.relationship_records())

**Filter/match by edge type** (use `select`)

Find entities connected by "Primary" relationship:

In [ ]:
crate.select(relationship_types="Primary")

You can find entities connected by multiple relationship types:

In [ ]:
crate.select(relationship_types=["Previous", "Related"])

Anything connected via "Previous" edge (source/target)

In [ ]:
crate2 = crate.pattern(via="preparedBy")

In [ ]:
crate2.glimpse()

In [ ]:
print(len(crate2), "entities")
# print(len(crate2.relationships), "relationships")

**Enrich / reshape edges**


Can't come up with a meaningful example of annotate_relationships()

In [ ]:
crate_simple = crate.collapse_edges()  # not sure how to show this so it's practically useful?

##  File content

The actual data files attached to file-entities.

>Going to leave this for now together with build_semantic_index()


In [ ]:
len(crate.files)

##  Graph manipulations (produces a new graph each time)

**Filter / subset** 

Select specific information in a graph. Select a certain entity type:

In [ ]:
people = crate.select(entity_types=["Person"])

...a certain time period:

In [ ]:
mid_cent = crate.select(time_range=(1945, 1960))

..or most connected entities:

In [ ]:
crate.select(min_connections=20)

You can also leave out certain information. Exclude a certain entity/relationship type:

In [ ]:
crate.exclude(relationship_types="Related")

...or drop some property values:

In [ ]:
crate.drop("#Elizabeth Daniels", property="preparedBy")

We can see that the .drop() method returns a graph with 4201 entities, while the opposite .where() method returns 69 entities. Together they form the original graph of 4270 entities.

In [ ]:
crate_ed = crate.where(preparedBy="#Elizabeth Daniels")

In [ ]:
crate_ed

You can take out entities of one graph from another one using subtract() method:

In [ ]:
crate.subtract(crate_ed)

Search (free text) for an entity in a graph. Default minimum rapidfuzz match score set to 80, but can be adjusted - as well as the number of entities displayed:

In [ ]:
crate.search("Elisabet")

In [ ]:
crate.search("Elisabeth", threshold=90, top_n=3)

**Transform** 


Raw string dates (like "1985", "c.1920s", or "1 March 2003") can be converted to ISO date strings:

In [ ]:
crate_dates = crate.convert_dates()

...or this can be performed on explicit fields of interest:

In [ ]:
crate.convert_dates(start="startDate", end="endDate")

Unsure how to use merge_nodes() meaningfully...?

In [ ]:
overview = crate.merge_nodes(by="type")
overwiew = crate.merge_nodes(by="organisation")

In [ ]:
overview.glimpse()

Add a community property to a graph:

In [ ]:
communities = crate.detect_communities()

In [ ]:
communities.visualise(colour_by="community", filepath="communities.html")

In [ ]:
!open "communities.html"

In [ ]:
communities.where(community=7)

**Visualising and exporting a graph**


Default renderer for exporting an interactive network in visualise() method is "2d" (sigma.js WebGL). Other options: "3d" 3d-force-graph, "svg" static SVG, "pyvis" pyvis/vis.js (needs extra install)

In [ ]:
crate.visualise(filepath="mycrate.html")

Open your network visualisation:

In [ ]:
!open "mycrate.html"

In [ ]:
crate.visualise(renderer="3d", filepath="3dcrate.html")

Not sure if colour_by/size_by is useful here...

In [ ]:
crate.visualise(colour_by="community", size_by="year", filepath="newcrate.html")

In [ ]:
!open "newcrate.html"

You can export a GraphML to visualise it in Gephi or Cytoscape:

In [ ]:
crate.write("crate_export.graphml", format="graphml")

Leaving these out as I don't quite know how to use/explain them:

crate.expand(...)                         
crate.query("(a:Person)-[:author]->(b)")   
crate.simplify()   
crate.layout()             # 2D node positions


detect_communities() is something that I don't quite find useful - feel free to delete/or lets discuss sometime

## Experiments


> **📝 Recipes:** _examples here_
